# Electoral Targeting Analysis of MPLADS Allocations

This notebook analyzes whether MPs strategically allocate MPLADS funds based on electoral considerations.

**Primary outcome variable:** Total recommended amount (Rs. in Crores) - captures actual resource allocation intent better than work counts.

**Key questions:**
1. Do MPs who won by narrow margins allocate more/faster?
2. Are there party-level differences in allocation patterns?
3. Do incumbent MPs allocate differently than first-termers?
4. How does constituency competitiveness affect allocation?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from pathlib import Path
import sys

sys.path.insert(0, str(Path(__file__).parent if "__file__" in dir() else Path(".")))
from _config import COLORS

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

DATA_DIR = Path("../data")

In [2]:
df = pd.read_csv(DATA_DIR / "mplads_election_merged.csv")
print(f"Total records: {len(df)}")
print(f"Matched with election data: {df['Constituency_Name'].notna().sum()}")
df.head()

Total records: 555
Matched with election data: 514


,state_name,constituency_name,mp_id,mp_name,total_works,works_recommended,works_sanctioned,works_completed,total_recommended_amount,total_actual_amount,...,Party_Type_TCPD,No_Terms,Incumbent,Recontest,MyNeta_education,TCPD_Prof_Main,state_clean,margin_category,party_category,alliance_2019
0,Andaman And Nicobar Islands,ANDAMAN AND NICOBAR ISLANDS,3042276,BISHNU PADA RAY,0,25,13,1,100724422.0,0.0,...,National Party,1.0,False,True,Graduate Professional,Business,ANDAMAN & NICOBAR ISLANDS,Very Close (<5%),National,UPA
1,Andhra Pradesh,AMALAPURAM(SC),3042277,GM Harish Balayogi,11,109,107,12,193442558.0,18123779.0,...,State-based Party,1.0,False,False,Graduate,Other,ANDHRA PRADESH,Very Close (<5%),Regional,Other
2,Andhra Pradesh,ANAKAPALLE,3042278,C.M.RAMESH,12,65,63,13,107175844.0,14747922.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Andhra Pradesh,ANANTAPUR,3042279,AMBICA G LAKSHMINARAYANA VALMIKI,25,140,115,26,209160362.0,18930181.0,...,State-based Party,1.0,False,False,Doctorate,Former Government,ANDHRA PRADESH,Safe (10-20%),Regional,Other
4,Andhra Pradesh,ARAKU(ST),3042280,Gumma Thanuja Rani,1,124,108,2,178300430.0,486815.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
matched = df[df["Constituency_Name"].notna()].copy()
print(f"Working with {len(matched)} MPs with election data")

Working with 514 MPs with election data


## 1. Summary Statistics

In [4]:
summary_vars = [
    "total_works", "works_recommended", "works_sanctioned", "works_completed",
    "total_recommended_amount", "completion_rate", "sanction_rate",
    "Vote_Share_Percentage", "Margin_Percentage", "No_Terms"
]
summary = matched[summary_vars].describe().round(2)

print("Amount in Crores:")
print(f"  Mean: {matched['total_recommended_amount'].mean() / 1e7:.2f} Cr")
print(f"  Median: {matched['total_recommended_amount'].median() / 1e7:.2f} Cr")
print(f"  Range: {matched['total_recommended_amount'].min() / 1e7:.2f} - {matched['total_recommended_amount'].max() / 1e7:.2f} Cr")
print()
summary

Amount in Crores:
  Mean: 15.16 Cr
  Median: 16.10 Cr
  Range: 0.00 - 62.01 Cr



,total_works,works_recommended,works_sanctioned,works_completed,total_recommended_amount,completion_rate,sanction_rate,Vote_Share_Percentage,Margin_Percentage,No_Terms
count,514.00,514.00,514.00,514.00,5.140000e+02,513.00,514.00,514.00,514.00,514.00
mean,40.40,163.53,124.10,41.40,1.516198e+08,0.32,0.75,52.64,17.34,1.96
std,64.18,147.07,123.55,64.19,6.303050e+07,0.28,0.24,7.64,12.24,1.35
min,0.00,1.00,0.00,0.00,0.000000e+00,0.00,0.00,32.17,0.02,1.00
25%,2.00,71.25,44.00,3.00,1.192212e+08,0.09,0.63,47.42,7.46,1.00
50%,19.00,129.00,89.00,20.00,1.609814e+08,0.25,0.82,52.32,15.10,1.50
75%,53.75,217.00,169.00,54.75,1.886430e+08,0.49,0.95,57.32,25.88,2.00
max,725.00,1456.00,1354.00,726.00,6.200518e+08,1.00,1.00,74.47,52.73,9.00


In [5]:
# Each MP gets 5 Cr per year = 25 Cr over 5 years (LS 18 term)
ENTITLEMENT_CR = 25.0

matched["completed_amount_cr"] = matched["total_actual_amount"] / 1e7
matched["recommended_pct"] = (matched["total_recommended_amount"] / 1e7 / ENTITLEMENT_CR) * 100
matched["completed_pct"] = (matched["completed_amount_cr"] / ENTITLEMENT_CR) * 100

print(f"Entitlement per MP (5-year term): Rs {ENTITLEMENT_CR} Cr")
print()
print("Recommended Amount (Cr):")
print(f"  Mean: {matched['total_recommended_amount'].mean() / 1e7:.2f} Cr ({matched['recommended_pct'].mean():.1f}% of entitlement)")
print(f"  Median: {matched['total_recommended_amount'].median() / 1e7:.2f} Cr")
print(f"  Range: {matched['total_recommended_amount'].min() / 1e7:.2f} - {matched['total_recommended_amount'].max() / 1e7:.2f} Cr")
print()
print("Completed Amount (Cr):")
print(f"  Mean: {matched['completed_amount_cr'].mean():.2f} Cr ({matched['completed_pct'].mean():.1f}% of entitlement)")
print(f"  Median: {matched['completed_amount_cr'].median():.2f} Cr")
print(f"  Range: {matched['completed_amount_cr'].min():.2f} - {matched['completed_amount_cr'].max():.2f} Cr")

Entitlement per MP (5-year term): Rs 25.0 Cr

Recommended Amount (Cr):
  Mean: 15.16 Cr (60.6% of entitlement)
  Median: 16.10 Cr
  Range: 0.00 - 62.01 Cr

Completed Amount (Cr):
  Mean: 1.94 Cr (7.8% of entitlement)
  Median: 1.10 Cr
  Range: 0.00 - 12.78 Cr


In [6]:
print("Party Distribution:")
print(matched["Party"].value_counts().head(15))
print("\nAlliance Distribution:")
print(matched["alliance_2019"].value_counts())

Party Distribution:
Party
BJP      285
INC       50
DMK       23
YSRCP     21
AITC      20
SHS       17
JD(U)     15
BJD       12
BSP       10
LJP        7
NCP        6
TRS        6
SP         5
IND        4
IUML       4
Name: count, dtype: int64

Alliance Distribution:
alliance_2019
NDA      294
Other    134
UPA       86
Name: count, dtype: int64


In [7]:
print("Margin Category Distribution:")
print(matched["margin_category"].value_counts())

Margin Category Distribution:
margin_category
Very Safe (>20%)    199
Safe (10-20%)       142
Very Close (<5%)     91
Close (5-10%)        82
Name: count, dtype: int64


## 2. Margin-Based Allocation Analysis

Do MPs who won by narrow margins allocate more funds?

In [ ]:
matched["amount_cr"] = matched["total_recommended_amount"] / 1e7

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(matched["Margin_Percentage"], matched["amount_cr"], alpha=0.5, color=COLORS[0])
ax.set_xlabel("Winning Margin (%)")
ax.set_ylabel("Recommended Amount (Cr)")
ax.set_title("Winning Margin vs Recommended Amount")

ax = axes[1]
margin_groups = matched.groupby("margin_category", observed=True).agg({
    "amount_cr": "mean",
    "constituency_name": "count"
}).reset_index()
margin_groups.columns = ["Margin Category", "Avg Amount (Cr)", "N"]
bars = ax.bar(margin_groups["Margin Category"], margin_groups["Avg Amount (Cr)"], color=COLORS[0])
ax.set_xlabel("Margin Category")
ax.set_ylabel("Avg Recommended Amount (Cr)")
ax.set_title("Average Amount by Margin Category")
for bar, n in zip(bars, margin_groups["N"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f"n={n}", ha="center")

plt.tight_layout()
plt.savefig(DATA_DIR / "fig_05_margin.png", dpi=150)
plt.show()

## 3. Party Patterns

In [9]:
party_stats = matched.groupby("Party").agg({
    "amount_cr": ["mean", "std"],
    "completion_rate": "mean",
    "mp_id": "count"
}).round(2)
party_stats.columns = ["Avg Amount (Cr)", "Std Amount (Cr)", "Completion Rate", "N MPs"]
party_stats = party_stats[party_stats["N MPs"] >= 5].sort_values("N MPs", ascending=False)
party_stats

,Avg Amount (Cr),Std Amount (Cr),Completion Rate,N MPs
Party,,,,
BJP,15.04,6.54,0.31,285
INC,15.97,5.21,0.30,50
DMK,17.87,4.78,0.44,23
YSRCP,13.17,7.40,0.30,21
AITC,16.27,8.29,0.38,20
SHS,10.26,5.69,0.41,17
JD(U),16.63,5.83,0.68,15
BJD,12.92,4.66,0.22,12
BSP,18.63,3.47,0.27,10


In [10]:
alliance_stats = matched.groupby("alliance_2019").agg({
    "amount_cr": "mean",
    "completion_rate": "mean",
    "mp_id": "count"
}).round(2)
alliance_stats.columns = ["Avg Amount (Cr)", "Completion Rate", "N MPs"]
alliance_stats

,Avg Amount (Cr),Completion Rate,N MPs
alliance_2019,,,
NDA,15.14,0.31,294
Other,14.58,0.36,134
UPA,16.16,0.33,86


In [ ]:
major_parties = ["BJP", "INC", "DMK", "YSRCP", "AITC", "SHS", "JD(U)", "BJD", "TDP"]
major_party_data = matched[matched["Party"].isin(major_parties)].copy()

fig, ax = plt.subplots(figsize=(12, 5))

party_means = major_party_data.groupby("Party")["amount_cr"].mean().sort_values(ascending=False)
ax.bar(party_means.index, party_means.values, color=COLORS[1])
ax.set_xlabel("Party")
ax.set_ylabel("Avg Recommended Amount (Cr)")
ax.set_title("Recommended Amount by Major Party")
ax.tick_params(axis='x', rotation=45)
bjp_mean = matched[matched["Party"] == "BJP"]["amount_cr"].mean()
ax.axhline(y=bjp_mean, color=COLORS[0], linestyle='--', label=f'BJP mean ({bjp_mean:.1f} Cr)')
ax.legend()

plt.tight_layout()
plt.savefig(DATA_DIR / "fig_06_party.png", dpi=150)
plt.show()

print("\nBJP vs Other Parties:")
bjp_stats = matched[matched["Party"] == "BJP"]["amount_cr"].mean()
other_stats = matched[matched["Party"] != "BJP"]["amount_cr"].mean()
print(f"  BJP:    Rs {bjp_stats:.2f} Cr avg")
print(f"  Others: Rs {other_stats:.2f} Cr avg")

## 4. Incumbent Behavior

In [12]:
incumbent_stats = matched.groupby("Incumbent").agg({
    "amount_cr": "mean",
    "completion_rate": "mean",
    "sanction_rate": "mean",
    "mp_id": "count"
}).round(2)
incumbent_stats.columns = ["Avg Amount (Cr)", "Completion Rate", "Sanction Rate", "N MPs"]
incumbent_stats

,Avg Amount (Cr),Completion Rate,Sanction Rate,N MPs
Incumbent,,,,
False,15.52,0.34,0.75,303
True,14.64,0.30,0.74,211


In [13]:
matched["terms_category"] = pd.cut(
    matched["No_Terms"],
    bins=[-1, 1, 2, 3, np.inf],
    labels=["First Term", "Second Term", "Third Term", "4+ Terms"]
)

terms_stats = matched.groupby("terms_category", observed=True).agg({
    "amount_cr": "mean",
    "completion_rate": "mean",
    "mp_id": "count"
}).round(2)
terms_stats.columns = ["Avg Amount (Cr)", "Completion Rate", "N MPs"]
terms_stats

,Avg Amount (Cr),Completion Rate,N MPs
terms_category,,,
First Term,15.44,0.33,257
Second Term,15.03,0.32,142
Third Term,14.96,0.37,50
4+ Terms,14.48,0.29,65


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
sns.boxplot(data=matched, x="terms_category", y="amount_cr", ax=ax, color=COLORS[2])
ax.set_xlabel("Number of Terms")
ax.set_ylabel("Recommended Amount (Cr)")
ax.set_title("Amount Distribution by Experience")

ax = axes[1]
valid_completion = matched[matched["completion_rate"].notna()]
sns.boxplot(data=valid_completion, x="terms_category", y="completion_rate", ax=ax, color=COLORS[2])
ax.set_xlabel("Number of Terms")
ax.set_ylabel("Completion Rate")
ax.set_title("Completion Rate by Experience")

plt.tight_layout()
plt.savefig(DATA_DIR / "fig_07_incumbent.png", dpi=150)
plt.show()

## 5. Competitiveness Effects

In [15]:
matched["competitive"] = matched["Margin_Percentage"] < 10

comp_stats = matched.groupby("competitive").agg({
    "amount_cr": "mean",
    "completion_rate": "mean",
    "unique_categories": "mean",
    "mp_id": "count"
}).round(2)
comp_stats.index = ["Safe Seat (margin >= 10%)", "Competitive (margin < 10%)"]
comp_stats.columns = ["Avg Amount (Cr)", "Completion Rate", "Category Diversity", "N MPs"]
comp_stats

,Avg Amount (Cr),Completion Rate,Category Diversity,N MPs
Safe Seat (margin >= 10%),14.91,0.35,1.64,341
Competitive (margin < 10%),15.66,0.27,1.75,173


In [16]:
print("ENOP (Effective Number of Parties) Analysis:")
matched["high_enop"] = matched["ENOP"] > matched["ENOP"].median()
enop_stats = matched.groupby("high_enop").agg({
    "amount_cr": "mean",
    "ENOP": "mean",
    "mp_id": "count"
}).round(2)
enop_stats.index = ["Low ENOP (< median)", "High ENOP (>= median)"]
enop_stats.columns = ["Avg Amount (Cr)", "Avg ENOP", "N MPs"]
enop_stats

ENOP (Effective Number of Parties) Analysis:


,Avg Amount (Cr),Avg ENOP,N MPs
Low ENOP (< median),14.57,2.17,285
High ENOP (>= median),15.90,2.77,229


## 6. Regression Analysis

In [17]:
reg_data = matched.copy()
reg_data["incumbent_flag"] = reg_data["Incumbent"].astype(int)
reg_data["female"] = (reg_data["Sex"] == "F").astype(int)

# Party dummies (BJP as reference category since it's the ruling party with most MPs)
reg_data["is_bjp"] = (reg_data["Party"] == "BJP").astype(int)
reg_data["is_inc"] = (reg_data["Party"] == "INC").astype(int)
reg_data["is_dmk"] = (reg_data["Party"] == "DMK").astype(int)
reg_data["is_ysrcp"] = (reg_data["Party"] == "YSRCP").astype(int)
reg_data["is_aitc"] = (reg_data["Party"] == "AITC").astype(int)
reg_data["is_shs"] = (reg_data["Party"] == "SHS").astype(int)
reg_data["is_jdu"] = (reg_data["Party"] == "JD(U)").astype(int)
reg_data["is_bjd"] = (reg_data["Party"] == "BJD").astype(int)
reg_data["is_tdp"] = (reg_data["Party"] == "TDP").astype(int)

print("Party distribution in regression data:")
print(reg_data["Party"].value_counts().head(10))

Party distribution in regression data:
Party
BJP      285
INC       50
DMK       23
YSRCP     21
AITC      20
SHS       17
JD(U)     15
BJD       12
BSP       10
LJP        7
Name: count, dtype: int64


In [18]:
# Model with major parties only (BJP as reference)
major_parties_mask = reg_data["Party"].isin(["BJP", "INC", "DMK", "YSRCP", "AITC", "SHS", "JD(U)", "BJD", "TDP"])
reg_major = reg_data[major_parties_mask].copy()

model1 = smf.ols(
    "amount_cr ~ Margin_Percentage + No_Terms + incumbent_flag + female + "
    "is_inc + is_dmk + is_ysrcp + is_aitc + is_shs + is_jdu + is_bjd + is_tdp + Turnout_Percentage",
    data=reg_major
).fit()
print(f"Model 1: Recommended Amount in Crores (BJP as reference, n={len(reg_major)} MPs from major parties)")
print(model1.summary().tables[1])

Model 1: Recommended Amount in Crores (BJP as reference, n=446 MPs from major parties)
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             18.6750      2.857      6.536      0.000      13.059      24.291
Margin_Percentage     -0.0502      0.026     -1.930      0.054      -0.101       0.001
No_Terms              -0.1018      0.288     -0.354      0.724      -0.667       0.464
incumbent_flag        -0.2838      0.846     -0.336      0.737      -1.946       1.379
female                 0.5113      0.882      0.580      0.562      -1.222       2.244
is_inc                 0.6331      1.039      0.609      0.543      -1.409       2.675
is_dmk                 3.1628      1.465      2.159      0.031       0.283       6.042
is_ysrcp              -1.9597      1.566     -1.252      0.211      -5.037       1.118
is_aitc                1.2817      1.627   

In [19]:
# Full model with all MPs, BJP as reference
model2 = smf.ols(
    "amount_cr ~ Margin_Percentage + No_Terms + incumbent_flag + female + "
    "is_inc + is_dmk + is_ysrcp + is_aitc + is_shs + is_jdu + is_bjd + is_tdp + Turnout_Percentage",
    data=reg_data
).fit()
print("Model 2: Recommended Amount in Crores (all MPs, BJP as reference)")
print(model2.summary().tables[1])

Model 2: Recommended Amount in Crores (all MPs, BJP as reference)
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             17.8112      2.299      7.746      0.000      13.294      22.329
Margin_Percentage     -0.0528      0.024     -2.189      0.029      -0.100      -0.005
No_Terms              -0.0852      0.250     -0.340      0.734      -0.577       0.407
incumbent_flag        -0.2406      0.741     -0.325      0.745      -1.696       1.214
female                 0.6454      0.807      0.800      0.424      -0.939       2.230
is_inc                 0.5085      0.980      0.519      0.604      -1.416       2.433
is_dmk                 3.0955      1.408      2.199      0.028       0.330       5.861
is_ysrcp              -2.1744      1.483     -1.466      0.143      -5.088       0.740
is_aitc                0.9932      1.537      0.646      0.518  

In [20]:
# Model 3: Log amount with party dummies (for robustness)
reg_data["log_amount"] = np.log1p(reg_data["total_recommended_amount"])
model3 = smf.ols(
    "log_amount ~ Margin_Percentage + No_Terms + incumbent_flag + female + "
    "is_inc + is_dmk + is_ysrcp + is_aitc + is_shs + is_jdu + is_bjd + is_tdp + Turnout_Percentage",
    data=reg_data
).fit()
print("Model 3: Log Recommended Amount (for robustness check)")
print(model3.summary().tables[1])

Model 3: Log Recommended Amount (for robustness check)
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             17.9599      1.030     17.438      0.000      15.936      19.983
Margin_Percentage     -0.0087      0.011     -0.809      0.419      -0.030       0.012
No_Terms               0.0476      0.112      0.424      0.671      -0.173       0.268
incumbent_flag        -0.2698      0.332     -0.813      0.416      -0.922       0.382
female                 0.2897      0.361      0.802      0.423      -0.420       1.000
is_inc                -0.1245      0.439     -0.284      0.777      -0.987       0.738
is_dmk                 0.4163      0.630      0.660      0.509      -0.822       1.655
is_ysrcp              -1.7929      0.664     -2.699      0.007      -3.098      -0.488
is_aitc               -0.7029      0.688     -1.021      0.308      -2.055 

In [21]:
# Summary table with party effects
results_df = pd.DataFrame({
    "Variable": model2.params.index,
    "Coef (Cr)": model2.params.round(2),
    "SE": model2.bse.round(2),
    "p-value": model2.pvalues.round(3),
    "Sig": (model2.pvalues < 0.05).map({True: "*", False: ""}) + (model2.pvalues < 0.01).map({True: "*", False: ""})
})
results_df = results_df.set_index("Variable")
print("Regression Results: Recommended Amount (Crores)")
print("(BJP is reference category - coefficients show Cr difference from BJP)")
print(results_df)

Regression Results: Recommended Amount (Crores)
(BJP is reference category - coefficients show Cr difference from BJP)
                    Coef (Cr)    SE  p-value Sig
Variable                                        
Intercept               17.81  2.30    0.000  **
Margin_Percentage       -0.05  0.02    0.029   *
No_Terms                -0.09  0.25    0.734    
incumbent_flag          -0.24  0.74    0.745    
female                   0.65  0.81    0.424    
is_inc                   0.51  0.98    0.604    
is_dmk                   3.10  1.41    0.028   *
is_ysrcp                -2.17  1.48    0.143    
is_aitc                  0.99  1.54    0.518    
is_shs                  -4.99  1.57    0.002  **
is_jdu                   1.25  1.69    0.459    
is_bjd                  -2.85  1.86    0.127    
is_tdp                   0.51  3.65    0.889    
Turnout_Percentage      -0.02  0.03    0.485    


## 7. Work Category Analysis

In [22]:
categories_all = []
for idx, row in matched.iterrows():
    if pd.notna(row["categories_list"]):
        for cat in str(row["categories_list"]).split("|"):
            categories_all.append({
                "mp_id": row["mp_id"],
                "category": cat.strip(),
                "margin_category": row["margin_category"],
                "alliance": row["alliance_2019"]
            })

cat_df = pd.DataFrame(categories_all)
print("Category Distribution:")
print(cat_df["category"].value_counts().head(10))

Category Distribution:
category
Normal/Others            503
Repair and Renovation    216
Trust and Society        134
Bar and Associations       9
Name: count, dtype: int64


In [23]:
cat_by_margin = pd.crosstab(
    cat_df["category"], 
    cat_df["margin_category"], 
    normalize="columns"
) * 100
cat_by_margin = cat_by_margin.round(1)
cat_by_margin.head(10)

margin_category,Close (5-10%),Safe (10-20%),Very Close (<5%),Very Safe (>20%)
category,,,,
Bar and Associations,0.7,1.4,1.3,0.9
Normal/Others,56.6,62.2,56.0,57.7
Repair and Renovation,29.4,24.3,27.7,22.5
Trust and Society,13.3,12.2,15.1,18.9


## 8. Key Findings Summary

In [24]:
print("="*60)
print("KEY FINDINGS SUMMARY")
print("="*60)

print("\n1. MARGIN-BASED ALLOCATION:")
margin_coef = model2.params.get("Margin_Percentage", 0)
margin_pval = model2.pvalues.get("Margin_Percentage", 1)
print(f"   - Coefficient on margin: {margin_coef:.3f} Cr (p={margin_pval:.3f})")
if margin_pval < 0.05:
    direction = "more" if margin_coef < 0 else "less"
    print(f"   - MPs in competitive seats allocate {direction} funds")
else:
    print("   - No significant relationship between margin and allocation")

print("\n2. PARTY PATTERNS (Cr difference from BJP):")
party_results = []
for party_var, party_name in [("is_inc", "INC"), ("is_dmk", "DMK"), ("is_ysrcp", "YSRCP"), 
                               ("is_aitc", "AITC"), ("is_shs", "SHS"), ("is_jdu", "JD(U)"),
                               ("is_bjd", "BJD"), ("is_tdp", "TDP")]:
    coef = model2.params.get(party_var, 0)
    pval = model2.pvalues.get(party_var, 1)
    sig = "*" if pval < 0.05 else ""
    party_results.append((party_name, coef, pval, sig))
    
party_results.sort(key=lambda x: abs(x[1]), reverse=True)
for party_name, coef, pval, sig in party_results:
    print(f"   - {party_name:6s}: {coef:+7.2f} Cr vs BJP (p={pval:.3f}) {sig}")

print("\n3. INCUMBENT EFFECT:")
inc_coef = model2.params.get("incumbent_flag", 0)
inc_pval = model2.pvalues.get("incumbent_flag", 1)
print(f"   - Incumbent coefficient: {inc_coef:.2f} Cr (p={inc_pval:.3f})")

print("\n4. EXPERIENCE EFFECT:")
terms_coef = model2.params.get("No_Terms", 0)
terms_pval = model2.pvalues.get("No_Terms", 1)
print(f"   - Per additional term: {terms_coef:.2f} Cr (p={terms_pval:.3f})")

print("\n5. GENDER EFFECT:")
female_coef = model2.params.get("female", 0)
female_pval = model2.pvalues.get("female", 1)
print(f"   - Female MP coefficient: {female_coef:.2f} Cr (p={female_pval:.3f})")

print("\n" + "="*60)
print(f"Model R-squared: {model2.rsquared:.3f}")
print(f"N observations: {int(model2.nobs)}")
print("="*60)

KEY FINDINGS SUMMARY

1. MARGIN-BASED ALLOCATION:
   - Coefficient on margin: -0.053 Cr (p=0.029)
   - MPs in competitive seats allocate more funds

2. PARTY PATTERNS (Cr difference from BJP):
   - SHS   :   -4.99 Cr vs BJP (p=0.002) *
   - DMK   :   +3.10 Cr vs BJP (p=0.028) *
   - BJD   :   -2.85 Cr vs BJP (p=0.127) 
   - YSRCP :   -2.17 Cr vs BJP (p=0.143) 
   - JD(U) :   +1.25 Cr vs BJP (p=0.459) 
   - AITC  :   +0.99 Cr vs BJP (p=0.518) 
   - TDP   :   +0.51 Cr vs BJP (p=0.889) 
   - INC   :   +0.51 Cr vs BJP (p=0.604) 

3. INCUMBENT EFFECT:
   - Incumbent coefficient: -0.24 Cr (p=0.745)

4. EXPERIENCE EFFECT:
   - Per additional term: -0.09 Cr (p=0.734)

5. GENDER EFFECT:
   - Female MP coefficient: 0.65 Cr (p=0.424)

Model R-squared: 0.053
N observations: 514
